In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from settings import MALL_CUSTOMERS

In [4]:
df = pd.read_csv(MALL_CUSTOMERS)
df.head()

,CustomerID,Gender,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40


In [6]:
df.isnull().sum()

CustomerID                0
Gender                    0
Age                       0
Annual Income (k$)        0
Spending Score (1-100)    0
dtype: int64

In [14]:
df['Gender'] = df.apply(lambda x : 0 if x['Gender'] == 'Female' else 1 , axis = 1)

In [15]:
X = df.drop(columns=['CustomerID'])
X.head()

,Gender,Age,Annual Income (k$),Spending Score (1-100)
0,1,19,15,39
1,1,21,15,81
2,0,20,16,6
3,0,23,16,77
4,0,31,17,40


In [16]:
X_train , X_test = train_test_split(X , test_size=0.2 , random_state=42)
print(X_train.shape, X_test.shape)

(160, 4) (40, 4)


In [17]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
X_test

array([[ 1.14840785, -1.07958231, -0.02219935,  0.00854281],
       [ 1.14840785, -1.22437122, -1.48689856,  1.03368009],
       [ 1.14840785,  1.52661807, -1.12072376, -1.81392346],
       [ 1.14840785, -0.35563776,  0.6369153 , -1.9278276 ],
       [ 1.14840785,  1.45422361,  0.38059294, -1.54814713],
       [-0.87077078, -1.44155458,  0.16088805, -0.06739328],
       [-0.87077078, -0.50042667, -0.46160911, -0.18129743],
       [ 1.14840785,  0.07872897,  0.96647262, -1.47221103],
       [-0.87077078,  0.94746243,  1.0030901 , -1.47221103],
       [-0.87077078, -1.07958231, -0.79116644,  0.50212742],
       [-0.87077078,  0.29591233, -0.46160911, -0.06739328],
       [ 1.14840785,  0.5130957 ,  1.36926491, -1.39627494],
       [-0.87077078, -0.21084885,  0.89323766,  0.8818079 ],
       [-0.87077078, -1.15197676, -0.24190423,  0.00854281],
       [-0.87077078,  1.09225134,  1.47911735, -1.05456251],
       [ 1.14840785, -0.86239894,  1.0030901 ,  0.65399961],
       [-0.87077078,  0.

In [28]:
class k_details:
    def __init__(self , final_centroids , labels , wcss):
        self.final_centroids = final_centroids
        self.labels = labels 
        self.wcss = wcss

In [29]:
def train(X, MAX_K=10, tol=1e-4, max_iter=100):

    all_models = {}   

    for k in range(1, MAX_K + 1):

       
        idxs = np.random.choice(X.shape[0], size=k, replace=False)
        centroids = X[idxs]

        for _ in range(max_iter):

          
            labels = {c: [] for c in range(k)}

           
            for data in X:
                distances = np.linalg.norm(data - centroids, axis=1)
                closest_idx = np.argmin(distances)
                labels[closest_idx].append(data)

            prev_centroids = centroids.copy()

          
            for c in range(k):
                if len(labels[c]) > 0:
                    centroids[c] = np.mean(labels[c], axis=0)

          
            shift = np.linalg.norm(centroids - prev_centroids)
            if shift < tol:
                break

        
        wcss = 0
        for c in range(k):
            for point in labels[c]:
                wcss += np.linalg.norm(point - centroids[c])**2

        all_models[k] = k_details(centroids.copy(), labels, wcss)

    return all_models


In [36]:
models = train(X_train, MAX_K=10)


for k in models:
    print(k, models[k].wcss)




1 640.0000000000002
2 473.31999060329815
3 385.8186768075088
4 330.9914780728648
5 278.19115350143153
6 252.93530137233424
7 213.02097312149198
8 174.33892992535291
9 169.22101901342504
10 123.99203808067723


In [37]:
def test(X_test, model):
    centroids = model.final_centroids
    k = centroids.shape[0]

    predictions = []

    for data in X_test:
        distances = np.linalg.norm(data - centroids, axis=1)
        closest_idx = np.argmin(distances)
        predictions.append(closest_idx)

    return np.array(predictions)


In [41]:
model_k2 = models[9]
preds = test(X_test, model_k2)
print(preds)



[2 7 3 2 0 6 6 2 1 6 5 2 8 6 1 4 5 1 0 2 6 5 1 1 2 0 6 1 8 6 7 3 0 2 2 0 8
 4 0 1]
